# Datamine MODTRI Process Example

This notebook demonstrates how to configure and run the **`modtri`** process wrapper in `dmstudio`.

## Process Description

## MODTRI

Convert the visible faces of a sub-cell block model to wireframe surfaces. The block model is interpreted as a solid model.

* **Note** (This process is much faster than the **[BLKTRI](<blktri.md>)** process and will work better on larger models.):

When using the @**PTTOL** parameter to detect duplicate wireframe output points the process will be slower

The @**ORIGIN** parameter can be used with more complex sub celled models to help resolve producing unverifiable wireframes.

If the block model is a standard rotated block model then the additional fields **X0, Y0, Z0, ANGLE1, ANGLE2, ANGLE3, ROTAXIS1, ROTAXIS2,** and **ROTAXIS3** are used to calculate the wireframe position.

The standard wireframe GROUP, SURFACE, adjacency and orientation data is output to the wireframe files. Normals for void surfaces will face inwards and can be used in Studio 3 evaluation processes to automatically exclude the evaluation of the void volume.

### Input Files:

* **in** (Block Model):
  Input model file. Must contain fields **XC, YC, ZC, XINC, YINC, ZINC, XMORIG, YMORIG,
  ZMORIG, NX, NY, NZ** , and **IJK**. If it is a Rotated Model then it must also include
  the fields **X0, Y0, Z0, ANGLE1, ANGLE2, ANGLE3, ROTAXIS1, ROTAXIS2** , and

## **ROTAXIS3**.

  Required=Yes

### Output Files:

* **wiretr** (Yes):
  Wireframe Triangle
  Required=No

* **wirept** (Yes):
  Wireframe Points
  Required=No

### Fields:

### Parameters:

* **origin**:
  Origin coordinates. Values are: 0 - retain original coordinates. 1 - use coordinates
  relative to the model origin. 2 - use the central coordinates of model cells as the new
  origin
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

* **pttol**:
  Check for duplicates in output wireframe coordinates
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('modtri')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute modtri
print("Running modtri...")
dm_cmd.modtri(
    in_i='t_assays',  # required
    # wiretr_o='t_modtri_out',  # optional
    # wirept_o='t_modtri_out',  # optional
    # origin_p=0,  # optional
    # pttol_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("modtri execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_modtri_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")